# Polars — E-commerce Business Transaction

Procesamiento distribuido del dataset de transacciones de comercio electrónico con **Polars**. Se ejecutan 10 consultas cubriendo limpieza, transformaciones, tratamiento de duplicados, filtrado, agregaciones, agrupaciones, ordenamiento y cálculo de métricas.

## Instalación de dependencias

In [1]:
!pip install -q polars
from pathlib import Path
import polars as pl

## Descarga del dataset

Si el dataset no está disponible localmente, se descarga desde Kaggle y se deposita en `dataset/`. Las rutas son relativas a la raíz del proyecto.

In [2]:
import kagglehub

DATASET_DIR = Path.cwd().parent / "dataset"
LOCAL_FILE = DATASET_DIR / "sales_transaction.csv"

if not LOCAL_FILE.exists():
    cached = Path(kagglehub.dataset_download("gabrielramos87/an-online-shop-business"))
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_FILE.write_bytes((cached / "Sales Transaction v.4a.csv").read_bytes())
    print(f"Dataset descargado en: {LOCAL_FILE}")
else:
    print(f"Dataset ya disponible: {LOCAL_FILE}")

Dataset ya disponible: /home/bricafio/Escritorio/BigData/dataset/sales_transaction.csv


## Lectura del dataset

In [3]:
df = pl.read_csv(
    LOCAL_FILE,
    schema_overrides={"TransactionNo": pl.String, "ProductNo": pl.String},
    null_values={"CustomerNo": "NA"},
)
print(f"Dimensiones: {df.shape}")

Dimensiones: (536350, 8)


## Consulta 1: Tratar nulos de CustomerNo

Rubro: Limpieza de datos

In [4]:
nulos = df.filter(pl.col("CustomerNo").is_null()).height
print(f"Filas con CustomerNo nulo: {nulos}")
df = df.filter(pl.col("CustomerNo").is_not_null())
print(f"Dimensiones tras eliminar nulos: {df.shape}")

Filas con CustomerNo nulo: 55
Dimensiones tras eliminar nulos: (536295, 8)


## Consulta 2: Eliminar duplicados

Rubro: Deduplicación

In [5]:
duplicados = df.height - df.unique(keep="first").height
print(f"Filas duplicadas: {duplicados}")
df = df.unique(keep="first")
print(f"Dimensiones tras eliminar duplicados: {df.shape}")

Filas duplicadas: 5200
Dimensiones tras eliminar duplicados: (531095, 8)


## Consulta 3: Convertir Date y extraer componentes

Rubro: Transformación de variables

In [6]:
df = df.with_columns(pl.col("Date").str.to_datetime("%m/%d/%Y").alias("Date"))
df = df.with_columns(
    pl.col("Date").dt.year().alias("Año"),
    pl.col("Date").dt.month().alias("Mes"),
    pl.col("Date").dt.day().alias("Día"),
)
print(df.select("Date", "Año", "Mes", "Día").head(3))

shape: (3, 4)
┌─────────────────────┬──────┬─────┬─────┐
│ Date                ┆ Año  ┆ Mes ┆ Día │
│ ---                 ┆ ---  ┆ --- ┆ --- │
│ datetime[μs]        ┆ i32  ┆ i8  ┆ i8  │
╞═════════════════════╪══════╪═════╪═════╡
│ 2019-07-29 00:00:00 ┆ 2019 ┆ 7   ┆ 29  │
│ 2019-11-08 00:00:00 ┆ 2019 ┆ 11  ┆ 8   │
│ 2019-11-08 00:00:00 ┆ 2019 ┆ 11  ┆ 8   │
└─────────────────────┴──────┴─────┴─────┘


## Consulta 4: Crear TotalSales

Rubro: Transformación

In [7]:
df = df.with_columns((pl.col("Price") * pl.col("Quantity")).alias("TotalSales"))
print(df.select("Price", "Quantity", "TotalSales").head(3))

shape: (3, 3)
┌───────┬──────────┬────────────┐
│ Price ┆ Quantity ┆ TotalSales │
│ ---   ┆ ---      ┆ ---        │
│ f64   ┆ i64      ┆ f64        │
╞═══════╪══════════╪════════════╡
│ 14.48 ┆ 5        ┆ 72.4       │
│ 16.18 ┆ 1        ┆ 16.18      │
│ 11.53 ┆ 1        ┆ 11.53      │
└───────┴──────────┴────────────┘


## Consulta 5: Filtrar cancelaciones

Rubro: Filtrado

In [8]:
filas = df.filter(
    (pl.col("Quantity") >= 0)
    & (~pl.col("TransactionNo").cast(pl.String).str.starts_with("C"))
)
print(f"Cancelaciones eliminadas: {df.height - filas.height}")
df = filas
print(f"Dimensiones tras filtrar: {df.shape}")

Cancelaciones eliminadas: 8494
Dimensiones tras filtrar: (522601, 12)


## Consulta 6: Facturación mensual

Rubro: Agrupación y agregación

In [9]:
(df.group_by(["Año", "Mes"])
 .agg(pl.col("TotalSales").sum().alias("Facturación"))
 .sort(["Año", "Mes"]))

Año,Mes,Facturación
i32,i8,f64
2018,12,4.3976e6
2019,1,4.5484e6
2019,2,3.3273e6
2019,3,4.3847e6
2019,4,3.5793e6
…,…,…
2019,8,4.7498e6
2019,9,6.6138e6
2019,10,7.2123e6


## Consulta 7: Ingreso por país

Rubro: Agregación

In [10]:
(df.group_by("Country")
 .agg(pl.col("TotalSales").sum().alias("IngresoTotal"))
 .sort("IngresoTotal", descending=True))

Country,IngresoTotal
str,f64
"""United Kingdom""",5.2347e7
"""Netherlands""",2.1516e6
"""EIRE""",1.7118e6
"""Germany""",1.3698e6
"""France""",1.3299e6
…,…
"""Lebanon""",5692.32
"""Brazil""",4652.27
"""RSA""",4259.83


## Consulta 8: Top 10 productos

Rubro: Agrupación y ordenamiento

In [11]:
(df.group_by(["ProductNo", "ProductName"])
 .agg(pl.col("Quantity").sum().alias("Unidades"))
 .sort("Unidades", descending=True)
 .head(10))

ProductNo,ProductName,Unidades
str,str,i64
"""23843""","""Paper Craft Little Birdie""",80995
"""23166""","""Medium Ceramic Top Storage Jar""",78033
"""22197""","""Popcorn Holder""",56902
"""84077""","""World War 2 Gliders Asstd Desi…",54951
"""85099B""","""Jumbo Bag Red Retrospot""",48375
"""85123A""","""Cream Hanging Heart T-Light Ho…",37937
"""21212""","""Pack Of 72 Retrospot Cake Case…",36492
"""84879""","""Assorted Colour Bird Ornament""",36394
"""23084""","""Rabbit Night Light""",30742


## Consulta 9: Top 10 clientes

Rubro: Agrupación, agregación y ordenamiento

In [12]:
(df.group_by("CustomerNo")
 .agg(pl.col("TotalSales").sum().alias("GastoAcumulado"))
 .sort("GastoAcumulado", descending=True)
 .head(10))

CustomerNo,GastoAcumulado
i64,f64
14646,2.1123e6
16446,1.0027e6
14911,914204.19
12415,900545.54
18102,897137.36
17450,891069.53
12346,840113.8
14156,694202.51
13694,646116.78


## Consulta 10: Estadísticos descriptivos

Rubro: Cálculo de métricas

In [13]:
df.select(
    pl.col('Price').mean().alias('Precio media'),
    pl.col('Price').median().alias('Precio mediana'),
    pl.col('Price').min().alias('Precio min'),
    pl.col('Price').max().alias('Precio max'),
    pl.col('Price').std().alias('Precio desv'),
    pl.col('Quantity').mean().alias('Cantidad media'),
    pl.col('Quantity').median().alias('Cantidad mediana'),
    pl.col('Quantity').min().alias('Cantidad min'),
    pl.col('Quantity').max().alias('Cantidad max'),
    pl.col('Quantity').std().alias('Cantidad desv'),
    pl.col('TotalSales').mean().alias('TotalSales media'),
    pl.col('TotalSales').median().alias('TotalSales mediana'),
    pl.col('TotalSales').min().alias('TotalSales min'),
    pl.col('TotalSales').max().alias('TotalSales max'),
    pl.col('TotalSales').std().alias('TotalSales desv'),
)

Precio media,Precio mediana,Precio min,Precio max,Precio desv,Cantidad media,Cantidad mediana,Cantidad min,Cantidad max,Cantidad desv,TotalSales media,TotalSales mediana,TotalSales min,TotalSales max,TotalSales desv
f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64
12.63716,11.94,5.13,660.62,7.965974,10.667492,4.0,1,80995,157.54242,120.132385,44.48,5.13,1.0027e6,1860.158603
